# ARY.01 242-d Feature Ablation (Colab T4)

Parallel companion to local RTX 3060 sweeps. Set **PART=0** on one runtime and **PART=1** on another.

## Setup (pick ONE method)

**Method A — Sidebar upload (recommended, no dialog issues)**
1. On PC: `powershell .\scripts\pack_colab_bundle.ps1` → creates `PRISM_colab.zip`
2. In Colab: click **Files** (left panel) → **Upload** → pick `PRISM_colab.zip` (wait until it shows under `/content/`)
3. Run **Step 1 — Install from /content zip** below

**Method B — Upload widget** — run the dedicated **Step 0 — Upload widget** cell only (by itself), pick zip, then Step 1.

**Method C — Google Drive** — upload zip to Drive, set `USE_DRIVE = True` in Step 1, set `DRIVE_ZIP_PATH`.

**Method D — Git** — set `GIT_URL` in Step 1 if you pushed the repo.

In [8]:
# @title Fix broken zip extract (run once if you see backslash filenames in /content)
# Windows-built zips unpack as flat files like data\aryan_splits\train.npz — this repairs it.
import shutil
import zipfile
from pathlib import Path

REPO = Path("/content/PRISM")
ZIP = Path("/content/PRISM_colab.zip")

shutil.rmtree(REPO, ignore_errors=True)
REPO.mkdir(parents=True)

with zipfile.ZipFile(ZIP) as zf:
    for info in zf.infolist():
        name = info.filename.replace("\\", "/").lstrip("/")
        if not name or name.endswith("/"):
            continue
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(zf.read(info.filename))

for p in Path("/content").iterdir():
    if "\\" in p.name and p.is_file():
        p.unlink()

print("OK:", sorted(p.name for p in REPO.iterdir()))
print("train.npz:", (REPO / "data/aryan_splits/train.npz").exists())

OK: ['data', 'requirements.txt', 'scripts', 'src']
train.npz: True


In [ ]:
# @title Step 0 — Upload widget ONLY (optional — run this cell alone, then Step 1)
# Skip this if you already uploaded PRISM_colab.zip via the left Files panel.
from google.colab import files
print("Choose PRISM_colab.zip or PRISM.zip…")
uploaded = files.upload()
print("Uploaded to /content:", list(uploaded.keys()))

In [9]:
# @title Step 1 — Install from /content zip, Drive, or git
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO = Path("/content/PRISM")
GIT_URL = ""  # @param {type:"string"}
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_ZIP_PATH = "PRISM_colab.zip"  # @param {type:"string"} path inside My Drive

REQUIRED = [
    REPO / "requirements.txt",
    REPO / "scripts" / "ablate_ary242_blocks.py",
    REPO / "data" / "aryan_splits" / "train.npz",
]


def verify_repo(root: Path) -> None:
    missing = [str(p.relative_to(root)) for p in REQUIRED if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "PRISM incomplete. Missing:\n  - " + "\n  - ".join(missing)
        )


def setup_from_zip(zip_path: Path) -> None:
    print(f"Unzipping {zip_path} → {REPO} …")
    if REPO.exists():
        shutil.rmtree(REPO)
    REPO.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            # Windows Compress-Archive uses backslashes in entry names
            name = info.filename.replace("\\", "/").lstrip("/")
            if not name or name.endswith("/"):
                continue
            target = REPO / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(info.filename))
    for p in Path("/content").iterdir():
        if "\\" in p.name and p.is_file():
            p.unlink()


def find_zip_in_content() -> Path | None:
    preferred = [
        Path("/content/PRISM_colab.zip"),
        Path("/content/PRISM.zip"),
    ]
    for p in preferred:
        if p.is_file():
            return p
    zips = sorted(Path("/content").glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    return zips[0] if zips else None


if REPO.is_dir() and (REPO / "requirements.txt").exists():
    print("PRISM already installed at", REPO)
else:
    zip_path = None
    if GIT_URL.strip():
        if REPO.exists():
            shutil.rmtree(REPO)
        r = subprocess.run(["git", "clone", "--depth", "1", GIT_URL.strip(), str(REPO)])
        if r.returncode != 0:
            raise RuntimeError(f"git clone failed: {GIT_URL}")
    elif USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = Path("/content/drive/MyDrive") / DRIVE_ZIP_PATH
        if not zip_path.is_file():
            raise FileNotFoundError(f"Not on Drive: {zip_path}")
        setup_from_zip(zip_path)
    else:
        zip_path = find_zip_in_content()
        if zip_path is None:
            raise FileNotFoundError(
                "No zip in /content. Do ONE of:\n"
                "  A) Left Files panel → Upload → PRISM_colab.zip to /content, re-run this cell\n"
                "  B) Run Step 0 (upload widget) first, then re-run this cell\n"
                "  C) Set USE_DRIVE=True and put zip on Google Drive\n"
                "  D) Set GIT_URL to your repo\n"
                f"\nCurrent /content: {[p.name for p in Path('/content').iterdir()]}"
            )
        setup_from_zip(zip_path)

verify_repo(REPO)
os.chdir(REPO)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Ready:", REPO)
print("CUDA:", __import__("torch").cuda.is_available())
print("Splits:", [p.name for p in (REPO / "data" / "aryan_splits").glob("*.npz")])

PRISM already installed at /content/PRISM
Ready: /content/PRISM
CUDA: True
Splits: ['test.npz', 'val.npz', 'train.npz']


In [10]:
# @title Step 2 — GPU + batch-size probe
!python scripts/estimate_ary242_resources.py --smoke-train

=== GPU ===
{
  "cuda": true,
  "name": "Tesla T4",
  "total_vram_gb": 14.56,
  "sm_count": 40
}

=== VRAM probe (current GPU) ===
{
  "d_state": 242,
  "batch_size": 128,
  "lookback": 20,
  "peak_vram_mb": 256.0,
  "n_params_m": 3.801,
  "ok": true
}

Recommended batch for this GPU @ d_state=242: 128

=== Profile table ===
  rtx3060_12gb: batch=128  peak=256.0 MB
  colab_t4_15gb: batch=128  peak=256.0 MB

=== Smoke train (1 epoch, d_state=242, batch=128) ===
{
  "num_features": 242,
  "n_params": 3800941,
  "train_sec": 1.1527326107025146,
  "test_binary_f1": 0.35555555555555557,
  "test_binary_fpr": 0.34049079754601225,
  "test_mitre_f1_macro": 0.4828953395167755,
  "best_val_score": 0.7100020253574756
}

Rough sweep estimate (12 variants × ~25 epochs): 5.8 min

Saved -> /content/PRISM/results/ary242_ablation/resource_estimate.json


In [11]:
# @title Step 3 — Config (PART 0 or 1)
PART = 0  # @param {type:"integer"}

ALL_VARIANTS = [
    "full_242", "minus_dead_meta", "minus_B_mean", "minus_B_std",
    "minus_B_dispersion", "minus_C_flags", "minus_D_proto", "minus_E_top_ports",
    "B_std_only", "B_mean_only", "meta_only_ACDE", "core_std_ACD",
    "lean_no_ports", "lean_no_mean", "dispersion_only_B", "ymt_like_lean",
]
MY_VARIANTS = ALL_VARIANTS[PART::2]
print("This runtime trains:", MY_VARIANTS)

This runtime trains: ['full_242', 'minus_B_mean', 'minus_B_dispersion', 'minus_D_proto', 'B_std_only', 'meta_only_ACDE', 'lean_no_ports', 'dispersion_only_B']


In [12]:
# @title Step 4 — Run ablation
import subprocess, sys
for v in MY_VARIANTS:
    print("\n===", v, "===")
    subprocess.check_call([sys.executable, "scripts/ablate_ary242_blocks.py", v])


=== full_242 ===

=== minus_B_mean ===

=== minus_B_dispersion ===

=== minus_D_proto ===

=== B_std_only ===

=== meta_only_ACDE ===

=== lean_no_ports ===

=== dispersion_only_B ===


In [13]:
# @title Step 5 — Download results
from google.colab import files
from pathlib import Path
out = Path("results/ary242_ablation/ablation.json")
if out.exists():
    files.download(str(out))
else:
    print("No ablation.json — check models/checkpoints/ary242_ablation/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>